# LangGraph 개발 노트북

**규칙: 이 노트북에 로직을 쓰지 않는다.**
`src/agent/` 의 모듈을 import 해서 실행·관찰·디버깅만 한다.
셀에서 로직이 3줄 넘게 자라면 즉시 모듈로 내린다.

준비: `uv sync` 후 `uv run python -m ipykernel install --user --name scaffold`

## 0. 오토리로드

`src/agent/*.py` 를 편집하면 다음 셀 실행 시 자동 반영된다.
커널 재시작 없이 개발하기 위한 전제 조건.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

# 노트북에서는 로컬 인메모리 체크포인터를 쓴다.
os.environ.setdefault("ENV", "dev")
os.environ.setdefault("POSTGRES_URL", "")
# 컨테이너 밖에서 붙을 때는 호스트에 노출된 포트로.
os.environ.setdefault("INFERENCE_BASE_URL", "http://localhost:8080/v1")

## 1. 그래프 조립

`build_graph()` 는 컴파일되지 않은 빌더를 돌려준다.
체크포인터를 여기서 주입하는 것이 개발/운영을 가르는 유일한 지점이다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

from agent.graph import build_graph
from agent.settings import get_settings

settings = get_settings()
graph = build_graph().compile(checkpointer=MemorySaver())
settings

## 2. 토폴로지 눈으로 확인

노트북이 가장 유리한 지점. 엣지를 잘못 연결하면 여기서 바로 보인다.

In [ ]:
print(graph.get_graph().draw_ascii())  # 의존성: grandalf

In [ ]:
# 이미지가 필요하면 (mermaid.ink 로 나가는 네트워크 호출)
# from IPython.display import Image
# Image(graph.get_graph().draw_mermaid_png())
print(graph.get_graph().draw_mermaid())

## 3. 노드를 그래프 없이 단독 호출

노드가 순수 함수이므로 그래프를 거치지 않고 바로 부를 수 있다.
라우팅 문제인지 노드 내부 문제인지 분리하는 가장 빠른 방법.

In [ ]:
from langchain_core.messages import HumanMessage

from agent.nodes import call_model, should_continue
from agent.tools import add_numbers, lookup_doc

print(await lookup_doc.ainvoke({"topic": "checkpointer"}))
print(await add_numbers.ainvoke({"a": 2, "b": 40}))

In [ ]:
# 실제 추론 서버가 떠 있어야 동작한다
state = {"messages": [HumanMessage(content="2 더하기 40은?")], "tool_iterations": 0}
update = await call_model(state)
update["messages"][-1]

## 4. 전체 실행

In [ ]:
config = {"configurable": {"thread_id": "nb-1"}, "recursion_limit": 25}

result = await graph.ainvoke(
    {"messages": [HumanMessage(content="checkpointer가 뭔지 문서에서 찾아줘")], "tool_iterations": 0},
    config,
)
for m in result["messages"]:
    m.pretty_print()

## 5. 노드별 상태 전이 관찰

`stream_mode="updates"` 는 각 노드가 무엇을 반환했는지 보여준다.
어느 노드가 상태를 예상과 다르게 덮어썼는지 찾을 때 쓴다.

In [ ]:
async for step in graph.astream(
    {"messages": [HumanMessage(content="1 더하기 1은?")], "tool_iterations": 0},
    {"configurable": {"thread_id": "nb-2"}},
    stream_mode="updates",
):
    for node, update in step.items():
        print(f"--- {node} ---")
        print(update)

## 6. 토큰 스트리밍

FastAPI 의 `/v1/chat/stream` 이 그대로 감싸는 것과 동일한 호출.

In [ ]:
async for chunk, meta in graph.astream(
    {"messages": [HumanMessage(content="한 문장으로 자기소개해줘")], "tool_iterations": 0},
    {"configurable": {"thread_id": "nb-3"}},
    stream_mode="messages",
):
    if chunk.content:
        print(chunk.content, end="", flush=True)

## 7. 체크포인트 되감기 (time travel)

`get_state_history()` 로 과거 체크포인트를 꺼내 그 지점부터 다시 실행한다.
프롬프트를 바꿔가며 특정 분기만 재현할 때 유용하다.

In [ ]:
history = [s for s in graph.get_state_history(config)]
for s in history:
    print(s.config["configurable"]["checkpoint_id"], "| next:", s.next,
          "| msgs:", len(s.values.get("messages", [])))

In [ ]:
# 두 번째로 오래된 체크포인트 지점부터 재실행
if len(history) > 2:
    replay_from = history[2].config
    replayed = await graph.ainvoke(None, replay_from)
    replayed["messages"][-1].pretty_print()

## 8. 운영과 동일한 경로로 확인

노트북에서 만족스러우면, 앱을 띄워 같은 그래프가 HTTP 뒤에서도 도는지 본다.
여기서 처음 드러나는 문제(동시성, 취소, 직렬화)가 배포 리스크의 전부다.

```bash
uv run pytest -q
uv run uvicorn app.main:app --reload --port 8000
```

In [ ]:
import httpx

async with httpx.AsyncClient(base_url="http://localhost:8000", timeout=60) as c:
    r = await c.post("/v1/chat", json={"user_prompt": "thread가 뭐야?"})
    print(r.status_code, r.json())